In [1]:
from vllm import AsyncLLMEngine, SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
import json



from pyngrok import ngrok
import nest_asyncio
import asyncio
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

In [2]:
engine_args = AsyncEngineArgs(model="cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0", tensor_parallel_size=8, max_model_len=4096, gpu_memory_utilization=0.3)

In [3]:
engine = AsyncLLMEngine.from_engine_args(engine_args)

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2024-06-04 14:00:52,926	INFO worker.py:1749 -- Started a local Ray instance.


INFO 06-04 14:00:54 llm_engine.py:161] Initializing an LLM engine (v0.4.3) with config: model='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', speculative_config=None, tokenizer='cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=8, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), seed=0, served_model_name=cpm-ai/Ocelot-Ko-self-instruction-10.8B-v1.0)
INFO 06-04 14:01:18 utils.py:618] Found nccl from library libnccl.so.2
INFO 06-04 14:01:18 pynccl.py:65] vLLM is using nccl==2.20.5
(RayWorkerWrapper pid=838700) INFO 06-04 14:01:18 utils.py:618] Found nccl from library libnccl.so.2
(RayWorkerWrap

In [83]:
sampling_params = SamplingParams(temperature=0.9, top_p=0.90, top_k=5, max_tokens=1024)

In [86]:

# 입력값 변수
talent = """
1. 끊임없는 열정으로 미래에 도전하는 인재
2. 창의와 혁신으로 세상을 변화시키는 인재
3. 정직과 바른 행동으로 역할과 책임을 다하는 인재
"""
user_answer = """
[ 혁신을 실천하는 회사 ]
LG CNS를 지원한 이유는 ‘혁신’을 실천할 수 있는 곳이기 때문입니다.
LG CNS는 기존의 코딩 방식이 아닌 MDD라는 혁신적인 개발 방식을 이용해 2013년 국내 최초로 전북은행의 차세대 시스템을 성공적으로 적용했습니다. 이어 광주은행, 카카오뱅크 등 많은 은행권 기업이 LG CNS의 MDD 방식으로 시스템을 구축하였습니다.
지난 2016년에는 IT 서비스 업계 최초로 IoT 국제 표준 'oneM2M' 인증을 획득해 다양한 산업 영역의 IoT 데이터 호환성 확보 기반을 다졌습니다. 또한, 지난 2월, 바르셀로나에서 열린 MWC에서 LG전자와 LG유플러스, 그리고 LG CNS가 함께 처음으로 5G 이동통신 스마트 팩토리 서비스를 시연했다는 소식도 접했습니다.
점차 사람이 통계적으로 관리하는 단계에서 자동화된 시스템을 구축하는 단계로 넘어가고 있습니다. 독보적인 능력으로 혁신을 알고 실천할 수 있는 LG CNS에서 제 역량을 쏟아 일조하고 싶습니다."
"""

# 프롬프트
prompt = f"""
당신은 주어진 인재상을 기준으로 사용자의 자기소개서를 효과적으로 수정하는 역할을 수행합니다. 이 과정을 따라 작업을 진행하세요.

[입력 정보]
인재상: {talent}
자기소개서: {user_answer}

[첨삭 과정]
1. 자기소개서의 각 문단을 신중하게 검토하여 주요 업적과 경험을 식별하고, 이를 인재상의 각 특성과 어떻게 연결할 수 있는지 평가하세요.
2. 인재상을 자세히 읽고, 각 특성이 요구하는 주요 요소들을 이해한 후, 지원자의 경험과 이를 연결하세요.
3. 각 인재상에 부합하는 경험과 성과를 강조할 부분을 식별하고, 인재상과 직접적인 연관이 없는 부분에는 연결 문장을 추가하여 자연스럽게 연계되도록 하세요.
4. 지원자의 능력과 인재상이 어떻게 일치하는지 명확히 보여줄 수 있는 구체적인 예시나 사례를 추가하세요.
5. 자기소개서의 전체적인 흐름을 유지하면서, 자연스러운 문장 구성과 일관된 문체로 인재상을 강조하도록 내용을 통합하세요.
6. 수정된 자기소개서를 처음부터 끝까지 읽어보며 자연스러움과 인재상의 반영 여부를 확인하고, 모든 인재상이 충분히 반영되었는지 검토하세요.

[첨삭]
:


"""
formatted_prompt = f"instruction: {prompt}\n output:"

example_input = {
"prompt": formatted_prompt,
"stream": False, # assume the non-streaming case
"request_id": talent,
}

# 비동기 추론
results_generator = engine.generate(
    example_input["prompt"],
    sampling_params,
    example_input["request_id"]
    
)

final_output = None
async for request_output in results_generator:
    # front단 실시간 추론을 위한 코드
    final_output = request_output

INFO 06-04 15:59:39 async_llm_engine.py:553] Received request 
INFO 06-04 15:59:39 async_llm_engine.py:553] 1. 끊임없는 열정으로 미래에 도전하는 인재
INFO 06-04 15:59:39 async_llm_engine.py:553] 2. 창의와 혁신으로 세상을 변화시키는 인재
INFO 06-04 15:59:39 async_llm_engine.py:553] 3. 정직과 바른 행동으로 역할과 책임을 다하는 인재
INFO 06-04 15:59:39 async_llm_engine.py:553] : prompt: 'instruction: \n당신은 주어진 인재상을 기준으로 사용자의 자기소개서를 효과적으로 수정하는 역할을 수행합니다. 이 과정을 따라 작업을 진행하세요.\n\n[입력 정보]\n인재상: \n1. 끊임없는 열정으로 미래에 도전하는 인재\n2. 창의와 혁신으로 세상을 변화시키는 인재\n3. 정직과 바른 행동으로 역할과 책임을 다하는 인재\n\n자기소개서: \n[ 혁신을 실천하는 회사 ]\nLG CNS를 지원한 이유는 ‘혁신’을 실천할 수 있는 곳이기 때문입니다.\nLG CNS는 기존의 코딩 방식이 아닌 MDD라는 혁신적인 개발 방식을 이용해 2013년 국내 최초로 전북은행의 차세대 시스템을 성공적으로 적용했습니다. 이어 광주은행, 카카오뱅크 등 많은 은행권 기업이 LG CNS의 MDD 방식으로 시스템을 구축하였습니다.\n지난 2016년에는 IT 서비스 업계 최초로 IoT 국제 표준 \'oneM2M\' 인증을 획득해 다양한 산업 영역의 IoT 데이터 호환성 확보 기반을 다졌습니다. 또한, 지난 2월, 바르셀로나에서 열린 MWC에서 LG전자와 LG유플러스, 그리고 LG CNS가 함께 처음으로 5G 이동통신 스마트 팩토리 서비스를 시연했다는 소식도 접했습니다.\n점차 사람이 통계적으로 관리하는 단계에서 자동화된 시스템을 구축하는 단계로 넘어가고 있습니다. 독보적인 능력으로 혁신을 알

In [87]:
print(final_output.outputs[0].text)



[미래를 향한 끊임없는 열정과 도전]
LG CNS를 지원한 이유는 회사의 끊임없는 열정과 미래에 대한 도전 정신 때문입니다. LG CNS는 MDD라는 혁신적인 개발 방식을 통해 2013년 국내 최초로 전북은행의 차세대 시스템을 성공적으로 적용하였고, 이후 광주은행과 카카오뱅크 등 많은 은행권 기업이 LG CNS의 MDD 방식을 채택했습니다. 이러한 혁신은 제가 끊임없이 도전하며 미래를 향해 나아가고자 하는 인재상과 일치합니다.

[창의와 혁신으로 세상을 변화시키는 인재]
또한, LG CNS는 IT 서비스 업계 최초로 IoT 국제 표준 'oneM2M' 인증을 획득하여 다양한 산업 영역의 IoT 데이터 호환성 확보 기반을 다졌습니다. 지난 2월 바르셀로나에서 열린 MWC에서 LG전자와 LG유플러스, 그리고 LG CNS가 함께 5G 이동통신 스마트 팩토리 서비스를 시연한 것은 회사의 창의와 혁신 정신을 잘 보여줍니다. 이는 제가 세상을 변화시키는 창의적 인재로서의 가능성을 발휘할 수 있는 환경을 제공합니다.

[정직과 바른 행동으로 역할과 책임을 다하는 인재]
저는 항상 정직과 바른 행동을 중요시하며, 제 역할과 책임을 다하는 인재로 성장해왔습니다. 이러한 자세가 LG CNS의 정직과 바른 행동을 중시하는 인재상과 잘 맞다고 생각합니다. LG CNS에서 제가 가진 능력과 열정을 발휘하여 회사의 혁신에 기여하고, 함께 성장해나가고 싶습니다.

결론적으로, LG CNS의 혁신적인 개발 방식, 창의와 혁신, 정직과 바른 행동은 제가 가진 인재상과 완벽하게 일치합니다. 저는 이러한 가치들을 바탕으로 LG CNS에서 끊임없이 도전하며, 세상을 변화시키고, 정직과 바른 행동으로 제 역할을 다하는 인재로 성장하고자 합니다.


: 